# T01 Production Quick Start

Enterprise-style onboarding: one YAML contract, logger at startup, inject into a small service, then log with `layer` + `context` (not raw f-strings).

**Run sections in order**: setup → imports → optional YAML peek → your service code → build logger from YAML → see files.

- Kernel: `.hydra_env`
- Start Jupyter from the **repo root**, or set `HYDRA_LOGGER_REPO`.
- Config: `examples/config/tutorial_t01_enterprise_layers.yaml` — `base_log_dir` + `log_dir_name` → `examples/logs/tutorials/`.

## Expect

- **Console**: `app` + `error` layers.
- **Files**: `t01_app.jsonl`, `t01_audit.log`, `t01_error.jsonl`.


## 1. Repo setup

Path + working directory.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path(os.environ.get("HYDRA_LOGGER_REPO", Path.cwd())).resolve()
_tut = REPO / "examples" / "tutorials"
if not (_tut / "utility").is_dir():
    raise FileNotFoundError(
        "Start Jupyter from the hydra-logger repo root, or set "
        "HYDRA_LOGGER_REPO to that directory (must contain examples/tutorials/utility/)."
    )
sys.path.insert(0, str(_tut))

from utility import notebook_bootstrap

repo_root = notebook_bootstrap()



## 2. Imports + paths

`CONFIG_PATH` = YAML file. `LOG_DIR` = where this tutorial writes files.

In [ ]:
from dataclasses import dataclass
from typing import Any

from hydra_logger import create_sync_logger
from hydra_logger.config.loader import load_logging_config

CONFIG_PATH = repo_root / "examples" / "config" / "tutorial_t01_enterprise_layers.yaml"
LOG_DIR = repo_root / "examples" / "logs" / "tutorials"


## 3. Load & inspect config (optional)

`load_logging_config` validates YAML before `create_sync_logger`. Re-run after you edit the YAML.

In [ ]:
cfg = load_logging_config(CONFIG_PATH, strict_unknown_fields=True)
print("layers:", list(cfg.layers.keys()))
print("default_level:", cfg.default_level)


## 4. Application code (DI)

Pass `logger` in; use `layer`, `context`, `extra` for structured events.

In [ ]:
@dataclass
class PaymentRequest:
    order_id: str
    user_id: str
    amount: float
    currency: str = "USD"


class PaymentService:
    def __init__(self, logger: Any) -> None:
        self.logger = logger

    def process(self, req: PaymentRequest) -> dict[str, Any]:
        self.logger.info(
            "payment request received",
            layer="app",
            context={"order_id": req.order_id, "user_id": req.user_id},
            extra={"amount": req.amount, "currency": req.currency},
        )
        if req.amount <= 0:
            self.logger.warning(
                "invalid payment amount",
                layer="audit",
                context={"order_id": req.order_id, "user_id": req.user_id},
                extra={"amount": req.amount},
            )
            raise ValueError("Amount must be > 0")
        if req.amount > 10000:
            self.logger.error(
                "payment exceeds policy threshold",
                layer="error",
                context={"order_id": req.order_id, "user_id": req.user_id},
                extra={"amount": req.amount, "threshold": 10000},
            )
            raise RuntimeError("Risk policy blocked this payment")
        self.logger.info(
            "payment approved",
            layer="app",
            context={"order_id": req.order_id},
            extra={"status": "approved"},
        )
        return {"ok": True, "order_id": req.order_id, "status": "approved"}


## 5. Create logger from YAML + run scenarios

More than one happy path + audit + error so console and files show real traffic.

In [ ]:
with create_sync_logger(
    config_path=CONFIG_PATH,
    strict_unknown_fields=True,
    name="t01-tutorial",
) as logger:
    svc = PaymentService(logger)
    svc.process(PaymentRequest(order_id="ORD-1001", user_id="U-1", amount=99.5))
    svc.process(PaymentRequest(order_id="ORD-1004", user_id="U-4", amount=42.0))
    svc.process(PaymentRequest(order_id="ORD-1005", user_id="U-5", amount=250.0))
    logger.info(
        "batch settlement completed",
        layer="app",
        context={"batch_id": "B-09"},
        extra={"items": 3},
    )
    try:
        svc.process(PaymentRequest(order_id="ORD-1002", user_id="U-2", amount=0))
    except ValueError:
        pass
    try:
        svc.process(PaymentRequest(order_id="ORD-1003", user_id="U-3", amount=15000))
    except RuntimeError:
        pass


## 6. Read what landed on disk

Re-run after **§5** to print tails of the tutorial log files.

In [ ]:
print("File output (last lines):")
for _name in ("t01_app.jsonl", "t01_audit.log", "t01_error.jsonl"):
    _p = LOG_DIR / _name
    print(f"--- {_name} ---")
    if _p.is_file():
        _lines = _p.read_text(encoding="utf-8").splitlines()
        for _line in _lines[-8:]:
            print(_line)
    else:
        print("(missing — run the logger cell first)")


**Customize**: edit `tutorial_t01_enterprise_layers.yaml` (layers, levels, destinations), then re-run **§3** → **§5** → **§6**.